In [6]:
import pandas as pd
import pytz
import datetime as dt
from sklearn.linear_model import LinearRegression
import joblib
from sklearn.preprocessing import StandardScaler

In [8]:
def preprocess_QuantAQ(file_path, start_date, end_date):
    sensor = pd.read_csv(file_path, parse_dates=['timestamp'])
    sensor = sensor[['timestamp_local','rh','temp','pm25']].dropna(subset=['pm25'])
    sensor.rename(columns={'timestamp_local':'time'}, inplace=True)
    
    sensor['time'] = pd.to_datetime(sensor['time'])
    est = pytz.timezone('US/Eastern')
    sensor['time'] = sensor['time'].dt.tz_convert(est)
    sensor['time'] = sensor['time'].dt.tz_localize(None)
    sensor['day'] = sensor['time'].dt.date
    sensor['dayhour'] = sensor['time'].dt.strftime('%Y-%m-%d %H')
    sensor = sensor[(sensor['time'] >= start_date) & (sensor['time'] <= end_date)]

    sensor = sensor[::-1].reset_index(drop=True)
    # train only on first 3/4 of data
    # split_index = i.int(len(sensor) * 0.75)
    # sensor = sensorloc[:split_index]
    return sensor


In [ ]:
def create_enhanced_features(df, hourly=True):
    features = df.copy()
  ##  a, b = 17.27, 237.7  # Magnus-Tetens constants (standard meteorology)
       ## alpha = ((a * features['temp']) / (b + features['temp'])) + np.log(features['rh']/100.0 + 0.001)
        ##features['dew_point'] = (b * alpha) / (a - alpha)
        # Vapor Pressure Deficit: "Drying power" of air - affects particle water content
        ##features['vpd'] = 0.611 * np.exp((17.27 * features['temp']) / (features['temp'] + 237.3)) * (1 - features['rh']/100)
    if hourly:
        # Original features
        features['pm25'] = df['shourlyPM25']
        features['rh'] = df['shourlyRH']
        features['temp'] = df['shourlyTemp']
        
        # Interaction terms - these capture how RH/temp affect bias differently at different PM levels
        features['pm25_x_rh'] = df['shourlyPM25'] * df['shourlyRH']
        features['pm25_x_temp'] = df['shourlyPM25'] * df['shourlyTemp']
        features['rh_x_temp'] = df['shourlyRH'] * df['shourlyTemp']
        
        # Polynomial terms - capture non-linear effects
        features['pm25_sq'] = df['shourlyPM25'] ** 2
        features['rh_sq'] = df['shourlyRH'] ** 2
        features['temp_sq'] = df['shourlyTemp'] ** 2
    else:
        # Original features
        features['pm25'] = df['sdailyPM25']
        features['rh'] = df['sdailyRH']
        features['temp'] = df['sdailyTemp']
        
        # Interaction terms - these capture how RH/temp affect bias differently at different PM levels
        features['pm25_x_rh'] = df['sdailyPM25'] * df['sdailyRH']
        features['pm25_x_temp'] = df['sdailyPM25'] * df['sdailyTemp']
        features['rh_x_temp'] = df['sdailyRH'] * df['sdailyTemp']
        
        # Polynomial terms - capture non-linear effects
        features['pm25_sq'] = df['sdailyPM25'] ** 2
        features['rh_sq'] = df['sdailyRH'] ** 2
        features['temp_sq'] = df['sdailyTemp'] ** 2
    
    return features

In [10]:
# returns a model that predicts daily pm25 from hourly sensor data
def train_hourly_model(sensor_collocation_data, h_regular):
    temp = sensor_collocation_data.groupby('dayhour').agg(
        shourlyPM25=('pm25', lambda x: x.mean(skipna=True)),
        shourlyRH=('rh', lambda x: x.mean(skipna=True)),
        shourlyTemp=('temp', lambda x: x.mean(skipna=True))
    ).reset_index()

    merged = pd.merge(h_regular, temp, on='dayhour').dropna()
    X = create_enhanced_features(merged)
    feature_cols = ['pm25', 'rh', 'temp', 'pm25_x_rh', 'pm25_x_temp', 'rh_x_temp', 'pm25_sq', 'rh_sq', 'temp_sq']
    # feature_cols = ['shourlyPM25', 'shourlyRH', 'shourlyTemp']
    target = merged['rhourlyPM25']

    # inner join (default) - only keep rows that are in both dataframes
    scalar = StandardScaler()
    X_scaled = scalar.fit_transform(X[feature_cols])
    model = LinearRegression().fit(X_scaled, target)
    # return LinearRegression().fit(merged[feature_cols], target)

    return {
        'model': model,
        'scaler': scalar,
        'feature_cols': feature_cols
    }

def train_daily_model(sensor_collocation_data, d_regular):
    temp = sensor_collocation_data.groupby('day').agg(
        sdailyPM25=('pm25', lambda x: x.mean(skipna=True)),
        sdailyRH=('rh', lambda x: x.mean(skipna=True)),
        sdailyTemp=('temp', lambda x: x.mean(skipna=True))
    ).reset_index()

    # inner join (default) - only keep rows that are in both dataframes
    merged = pd.merge(temp, d_regular, on='day').dropna().sort_values('day')
    X = create_enhanced_features(merged, hourly=False)
    feature_cols = ['pm25', 'rh', 'temp', 'pm25_x_rh', 'pm25_x_temp', 
                       'rh_x_temp', 'pm25_sq', 'rh_sq', 'temp_sq']
    # feature_cols = ['sdailyPM25', 'sdailyRH', 'sdailyTemp']
    target = merged['rdailyPM25']

    scalar = StandardScaler()
    X_scaled = scalar.fit_transform(X[feature_cols])
    model = LinearRegression().fit(X_scaled, target)
    # return LinearRegression().fit(merged[feature_cols], target)

    return {
        'model': model,
        'scaler': scalar,
        'feature_cols': feature_cols
    }



In [11]:
quantAQ00589 = preprocess_QuantAQ("../ShortenedData/MOD-00589-RAW.csv", '2025-05-31', '2025-07-14')
quantAQ00590 = preprocess_QuantAQ("../ShortenedData/MOD-00590-RAW.csv", '2025-05-31', '2025-07-14')
quantAQ00591 = preprocess_QuantAQ("../ShortenedData/MOD-00591-RAW.csv", '2025-05-31', '2025-07-14')
quantAQ00592 = preprocess_QuantAQ("../ShortenedData/MOD-00592-RAW.csv", '2025-05-31', '2025-07-14')
quantAQ00593 = preprocess_QuantAQ("../ShortenedData/MOD-00593-RAW.csv", '2025-05-31', '2025-07-14')
quantAQ00595 = preprocess_QuantAQ("../ShortenedData/MOD-00595-RAW.csv", '2025-05-31', '2025-07-14')

In [12]:
# load and clean GAPA data
gapa = pd.read_csv('PM10&PM2.5.csv', header=2)
gapa.columns = ['Date', 'PMHR_2', 'PMHR', 'PMHR10', '24H_PMHR10', 'PMHRC', 'pm25']
gapa = gapa[['Date', 'PMHR']]

gapa['time'] = pd.to_datetime(gapa['Date'], format='%d-%b-%Y %H:%M')
gapa['day'] = gapa['time'].dt.date
gapa['dayhour'] = gapa['time'].dt.strftime("%Y-%m-%d %H")
gapa.rename(columns={'PMHR':'rpm25'}, inplace=True)

gapa = gapa.sort_values('time').reset_index(drop=True).dropna(subset=['rpm25'])

# train only on first 3/4 of data
# split_index = int(len(gapa) * 0.75)
# gapa = gapa.iloc[:split_index]

h_gapa = gapa.groupby('dayhour').agg(
    rhourlyPM25=('rpm25', lambda x: x.mean(skipna=True)),
).reset_index()

d_gapa = gapa.groupby('day').agg(
    rdailyPM25=('rpm25', lambda x: pd.to_numeric(x, errors='coerce').mean(skipna=True)),
).reset_index()




In [13]:
hmodel_00589 = train_hourly_model(quantAQ00589, h_gapa)
dmodel_00589 = train_daily_model(quantAQ00589, d_gapa)
hmodel_00590 = train_hourly_model(quantAQ00590, h_gapa)
dmodel_00590 = train_daily_model(quantAQ00590, d_gapa)
hmodel_00591 = train_hourly_model(quantAQ00591, h_gapa)
dmodel_00591 = train_daily_model(quantAQ00591, d_gapa)
hmodel_00592 = train_hourly_model(quantAQ00592, h_gapa)
dmodel_00592 = train_daily_model(quantAQ00592, d_gapa)
hmodel_00593 = train_hourly_model(quantAQ00593, h_gapa)
dmodel_00593 = train_daily_model(quantAQ00593, d_gapa)
hmodel_00595 = train_hourly_model(quantAQ00595, h_gapa)
dmodel_00595 = train_daily_model(quantAQ00595, d_gapa)

In [ ]:
# save models
joblib.dump(hmodel_00589, 'models/hmodel_00589.joblib')
joblib.dump(dmodel_00589, 'models/dmodel_00589.joblib')
joblib.dump(hmodel_00590, 'models/hmodel_00590.joblib')
joblib.dump(dmodel_00590, 'models/dmodel_00590.joblib')
joblib.dump(hmodel_00591, 'models/hmodel_00591.joblib')
joblib.dump(dmodel_00591, 'models/dmodel_00591.joblib')
joblib.dump(hmodel_00592, 'models/hmodel_00592.joblib')
joblib.dump(dmodel_00592, 'models/dmodel_00592.joblib')
joblib.dump(hmodel_00593, 'models/hmodel_00593.joblib')
joblib.dump(dmodel_00593, 'models/dmodel_00593.joblib')
joblib.dump(hmodel_00595, 'models/hmodel_00595.joblib')
joblib.dump(dmodel_00595, 'models/dmodel_00595.joblib')
